# SABR Calibration for Interest Rate Swaptions

The SABR model describes how implied volatility varies across strikes. Black's model assumes one
volatility per expiry; real swaption markets trade a smile. SABR reproduces that smile by letting
both the forward rate and its own volatility diffuse, with a correlation between them.

This notebook calibrates SABR slice by slice, validates the implementation against one day of real
VCUB quotes, prices a swaption with Black-76 under an annuity built from the SOFR curve of the same
date, and computes Greeks. Where no VCUB quotes exist it falls back to a FRED derived ATM level with
a synthetic smile, and every result carries a source tag so the two are never confused.

The snapshot is USD SOFR, 2026-07-28 mid: 13 expiries by 7 tenors, nine strikes each, 819 quotes in
normal (Bachelier) vol.

## The model

$$dF = \alpha \, F^{\beta} \, dW_1, \qquad d\alpha = \nu \, \alpha \, dW_2, \qquad \text{corr}(dW_1, dW_2) = \rho$$

| Parameter | Role |
|-----------|------|
| $\alpha$ | overall vol level, sets ATM |
| $\beta$ | backbone, how vol scales with the rate level. fixed at 0.5 here |
| $\nu$ | vol of vol, width of the smile wings |
| $\rho$ | correlation, direction and steepness of the skew |

$\beta$ is fixed rather than fitted. Calibrating all four parameters at once is unstable, since
$\beta$ and $\rho$ both act on the skew.

## Setup and synthetic surface

A swaption vol surface is a grid of implied vols by option expiry and strike, for each underlying
swap tenor. The synthetic surface below is a stand in used to exercise the machinery before any
real data is loaded. It is not market data and is labelled as such throughout.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import warnings

from sabr_swaption import (
    DARK_THEME, DEFAULT_OFFSETS_BP, SOURCE_LABELS,
    hagan_implied_vol, hagan_normal_vol, calibrate_sabr,
    black_price, sabr_price, compute_greeks, price_swaption,
    generate_sample_vol_surface, build_smile_from_atm, build_fallback_slice,
    load_vcub_snapshot, fred_atm_level, fetch_fred_par_curve, load_par_curve,
    bootstrap_discount_curve, discount_factor, curve_max_time,
    annuity_from_curve, forward_swap_rate, flat_annuity,
    parse_term, source_label,
)

warnings.filterwarnings('ignore')
plt.rcParams.update(DARK_THEME)

surface, expiries, tenors = generate_sample_vol_surface()

example = surface[(5, 10)]
print('synthetic example: 5Y expiry into 10Y swap')
print(f"  atm forward: {example['atm_forward']*100:.2f}%")
print(f"  strikes:     {np.round(example['strikes'] * 100, 2)} (%)")
print(f"  impl vols:   {np.round(example['implied_vols'] * 100, 2)} (%)")

In [ ]:
example = surface[(5, 10)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(example['strike_offsets_bp'], example['implied_vols'] * 100,
        'o-', color='#00d2ff', markersize=8, linewidth=2, label='vols')
ax.axvline(0, color='#ff6b6b', linestyle='--', alpha=0.6, label='ATM')
ax.set_xlabel('strike offset from ATM (bp)')
ax.set_ylabel('implied volatility (%)')
ax.set_title('Vol smile, 5Y into 10Y [SYNTHETIC, not market data]',
             fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Hagan's approximation

The SABR SDEs have no closed form implied vol, but Hagan, Kumar, Lesniewski and Woodward (2002)
give an asymptotic expansion mapping $(\alpha, \beta, \nu, \rho)$ to a Black implied vol for any
strike. It is fast and differentiable, which is what makes least squares calibration practical.

Two conventions matter here. `hagan_implied_vol` returns lognormal (Black) vol, used for the
synthetic surface and for pricing. `hagan_normal_vol` returns normal (Bachelier) vol in rate units,
which is the convention the VCUB snapshot is quoted in. Both take the same SABR parameters.

In [ ]:
test_strikes = np.linspace(0.01, 0.07, 100)
test_vols = hagan_implied_vol(test_strikes, 0.04, 5.0, 0.035, 0.5, 0.5, -0.3)

print(f"ATM lognormal vol: {float(hagan_implied_vol(0.04, 0.04, 5.0, 0.035, 0.5, 0.5, -0.3))*100:.2f}%")
print(f"ATM normal vol:    {float(hagan_normal_vol(0.04, 0.04, 5.0, 0.035, 0.5, 0.5, -0.3))*10000:.2f}bp")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(test_strikes * 100, test_vols * 100, color='#00d2ff', linewidth=2)
ax.axvline(4.0, color='#ff6b6b', linestyle='--', alpha=0.6, label='ATM = 4%')
ax.set_xlabel('strike (%)')
ax.set_ylabel('implied volatility (%)')
ax.set_title('SABR smile from the Hagan formula', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The left wing sits above the right. That asymmetry is the skew, driven by negative $\rho$. Setting
$\rho = 0$ makes the smile symmetric; raising $\nu$ widens the wings.

## Calibration

With $\beta$ fixed, calibration minimises squared vol error over the three remaining parameters:

$$\min_{\alpha, \nu, \rho} \sum_{i} \left( \sigma^{\text{mkt}}_i - \sigma^{\text{SABR}}_i(\alpha, \beta, \nu, \rho; K_i, F, T) \right)^2$$

subject to $\alpha > 0$, $\nu > 0$, $|\rho| < 1$. `calibrate_sabr` uses L-BFGS-B with box bounds and
seeds $\alpha$ from the ATM vol, $\nu = 0.5$, $\rho = -0.3$.

In [ ]:
data = surface[(5, 10)]
result = calibrate_sabr(data['strikes'], data['atm_forward'], 5.0, data['implied_vols'], beta=0.5)

print('calibrated to the SYNTHETIC 5Y into 10Y slice:')
print(f"  alpha = {result['alpha']:.6f}")
print(f"  beta  = {result['beta']:.1f} (fixed)")
print(f"  nu    = {result['nu']:.4f}")
print(f"  rho   = {result['rho']:.4f}")
print(f"  rmse  = {result['rmse']*10000:.2f} bp   converged = {result['converged']}")

fine_strikes = np.linspace(data['strikes'][0], data['strikes'][-1], 200)
fine_vols = hagan_implied_vol(fine_strikes, data['atm_forward'], 5.0,
                              result['alpha'], result['beta'], result['nu'], result['rho'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(data['strike_offsets_bp'], data['implied_vols'] * 100,
        'o', color='#00d2ff', markersize=9, label='input vols', zorder=5)
ax.plot((fine_strikes - data['atm_forward']) * 10000, fine_vols * 100,
        '-', color='#ff6b6b', linewidth=2.5, label='SABR fit')
ax.axvline(0, color='#888', linestyle=':', alpha=0.4)
ax.set_xlabel('strike offset from ATM (bp)')
ax.set_ylabel('implied volatility (%)')
ax.set_title('SABR calibration, 5Y into 10Y [SYNTHETIC, not market data]',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Parameters across the synthetic surface

Calibrating every (expiry, tenor) pair shows how the parameters move across the term structure.
Still synthetic data at this stage.

In [ ]:
calibration_results = {}

print(f"{'expiry':>6} {'tenor':>5} {'alpha':>8} {'nu':>8} {'rho':>8} {'rmse(bp)':>9} {'ok':>4}")
print('-' * 52)

for expiry in expiries:
    for tenor in tenors:
        data = surface[(expiry, tenor)]
        res = calibrate_sabr(data['strikes'], data['atm_forward'],
                             float(expiry), data['implied_vols'], beta=0.5)
        calibration_results[(expiry, tenor)] = res
        print(f"{expiry:>4}Y {tenor:>3}Y {res['alpha']:>8.5f} {res['nu']:>8.4f} "
              f"{res['rho']:>8.4f} {res['rmse']*10000:>8.2f} {'Y' if res['converged'] else 'N':>4}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
param_labels = [('alpha', 'alpha (vol level)'), ('nu', 'nu (vol of vol)'), ('rho', 'rho (skew)')]

for ax, (pname, plabel) in zip(axes, param_labels):
    for tenor in tenors:
        values = [calibration_results[(exp, tenor)][pname] for exp in expiries]
        ax.plot(expiries, values, 'o-', label=f'{tenor}Y tenor', linewidth=2, markersize=7)
    ax.set_xlabel('option expiry (years)')
    ax.set_ylabel(pname)
    ax.set_title(plabel, fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('SABR parameter term structure [SYNTHETIC]', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Greeks

Greeks come from chaining the two pieces already built: Hagan maps parameters to an implied vol,
Black-76 maps that vol to a price. `compute_greeks` bumps the inputs and differences the prices.

Vega is per 1bp move in $\alpha$, gamma is the second derivative in the forward, and vanna is the
cross term $\partial^2 P / \partial F \partial \sigma$, which is what tells a desk how its vol
exposure shifts as rates move.

In [ ]:
data = surface[(5, 10)]
res = calibration_results[(5, 10)]

greeks_by_strike = [compute_greeks(data['atm_forward'], K, 5.0,
                                   res['alpha'], res['beta'], res['nu'], res['rho'])
                    for K in data['strikes']]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
panels = [('price', 'price', '#00d2ff'), ('vega', 'vega (per 1bp)', '#ff6b6b'),
          ('gamma', 'gamma', '#ffd93d'), ('vanna', 'vanna', '#a855f7')]

for ax, (key, label, color) in zip(axes.flat, panels):
    ax.plot(data['strike_offsets_bp'], [g[key] for g in greeks_by_strike],
            'o-', color=color, linewidth=2, markersize=7)
    ax.axvline(0, color='#888', linestyle=':', alpha=0.4)
    ax.set_xlabel('strike offset (bp)')
    ax.set_ylabel(label)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('SABR Greeks, 5Y into 10Y payer [SYNTHETIC]', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Vega and gamma both peak near ATM. Vanna changes sign there, which is why a book with skew exposure
has to be watched on both axes at once.

---

# VCUB anchor layer

Everything above runs on synthetic data. This section loads the one day of real VCUB data and uses
it to check that the implementation reproduces a real smile.

## Loading the snapshot

The file is 819 Bloomberg VCUB quotes from 2026-07-28, USD SOFR mid, taken off the OTM
Swaptions/SABR tab as absolute normal vol: 91 smiles of nine strikes each, spanning ATM minus 200bp
to ATM plus 200bp, with the forward swap rate for each slice bootstrapped from the IRSB SOFR curve
of the same date.

Quotes are **normal (Bachelier)** vol, stored in decimal rate units, so 0.008922 is 89.22bp. The
`vol_model` column carries that convention through to calibration, which is why the notebook has
both `hagan_normal_vol` and the lognormal `hagan_implied_vol`.

The loader also accepts a per slice calibrated parameter file, in which case it reconstructs each
smile from its parameters and the validation section says so. If the file is missing entirely it
prints a message and returns an empty dict, and the notebook runs on synthetic data alone.

In [ ]:
vcub_snapshot = load_vcub_snapshot('data/vcub_snapshot.csv')

if vcub_snapshot:
    first = next(iter(vcub_snapshot))
    sl = vcub_snapshot[first]
    print(f"\nsnapshot date: {sl['date']}")
    print(f"quote kind:    {sl['quote_kind']}")
    print(f"quotes:        {sum(len(v['strikes']) for v in vcub_snapshot.values())}")
    print(f"vol convention: {sl['vol_model']}")
    print(f"slices:        {len(vcub_snapshot)}")
    print(f"\nexample slice {first[0]} into {first[1]}:")
    print(f"  atm forward: {sl['atm_forward']*100:.4f}%")
    print(f"  strikes:     {np.round(sl['strikes']*100, 3)} (%)")
    print(f"  strike offsets: {np.round(sl['strike_offsets_bp']).astype(int)} (bp)")
    print(f"  normal vols: {np.round(sl['implied_vols']*10000, 2)} (bp)")
else:
    print('\nno VCUB data available, downstream sections use fallback data only')

## Validation

For each VCUB slice: calibrate SABR to the quoted vols, report RMSE and max error in basis points,
and plot the fit against the quotes. A slice passes at RMSE under 5bp.

This is the real test. The inputs are dealer quotes at nine strikes, and nothing about them came
out of this code, so a tight fit says that Hagan's normal expansion with three free parameters
reproduces the smile the market actually traded. Where SABR struggles it will show up here rather
than being absorbed into the fit.

If the snapshot is a calibrated parameter file instead of raw quotes, the smile fed to the
optimiser was itself generated from SABR parameters, and a low RMSE would only show the
implementation is self consistent. The banner states which of the two ran.

In [ ]:
VALIDATION_RMSE_LIMIT_BP = 5.0
# set to None to plot every slice
VALIDATION_PLOT_SLICES = [('1Mo', '1Yr'), ('1Yr', '5Yr'), ('5Yr', '10Yr'),
                          ('10Yr', '10Yr'), ('20Yr', '10Yr'), ('30Yr', '30Yr')]
# the six above are a spread across the grid. set to None to plot all 91

validation_results = {}

if not vcub_snapshot:
    print('no VCUB snapshot, validation skipped')
else:
    reconstructed = 'reconstructed' in next(iter(vcub_snapshot.values()))['quote_kind']

    has_ref = 'ref_rmse_bp' in next(iter(vcub_snapshot.values()))
    header = (f"{'expiry':>7} {'tenor':>6} {'alpha':>9} {'nu':>8} {'rho':>8} "
              f"{'rmse(bp)':>9} {'max(bp)':>8}" + (f" {'ref rmse':>9}" if has_ref else ''))
    print(header)
    print('-' * len(header))

    for key, sl in vcub_snapshot.items():
        res = calibrate_sabr(sl['strikes'], sl['atm_forward'], sl['expiry_years'],
                             sl['implied_vols'], beta=0.5, vol_model=sl['vol_model'])
        res['source'] = 'VCUB'
        res['date'] = sl['date']
        res['quote_kind'] = sl['quote_kind']
        validation_results[key] = res
        line = (f"{key[0]:>7} {key[1]:>6} {res['alpha']:>9.6f} {res['nu']:>8.4f} "
                f"{res['rho']:>8.4f} {res['rmse']*10000:>9.3f} {res['max_err']*10000:>8.3f}")
        if has_ref:
            line += f" {sl['ref_rmse_bp']:>9.3f}"
        print(line)

    rmses_bp = np.array([r['rmse'] for r in validation_results.values()]) * 10000
    worst = max(validation_results, key=lambda k: validation_results[k]['rmse'])
    passed = rmses_bp.max() < VALIDATION_RMSE_LIMIT_BP

    banner = '\033[42m\033[30m' if passed else '\033[41m\033[37m'
    reset = '\033[0m'
    print()
    print(banner + ' ' * 78 + reset)
    if passed:
        print(banner + '  \u2705 code validated against real VCUB data'.ljust(77) + reset)
    else:
        print(banner + f"  FLAGGED  SABR fit exceeds {VALIDATION_RMSE_LIMIT_BP:.0f}bp on at least one VCUB slice".ljust(78) + reset)
    print(banner + f"  {len(validation_results)} VCUB slices, max RMSE {rmses_bp.max():.3f}bp "
                   f"({worst[0]} into {worst[1]}), mean {rmses_bp.mean():.3f}bp".ljust(78) + reset)
    if reconstructed:
        print(banner + '  scope: implementation self consistency and parameter recovery.'.ljust(78) + reset)
        print(banner + '  NOT a market fit test, the input smile came from VCUB SABR parameters.'.ljust(78) + reset)
    else:
        n_over = int((rmses_bp >= VALIDATION_RMSE_LIMIT_BP).sum())
        print(banner + '  scope: SABR fit to raw VCUB strike quotes, a genuine market fit test.'.ljust(78) + reset)
        print(banner + f'  {n_over} of {len(rmses_bp)} slices at or above the {VALIDATION_RMSE_LIMIT_BP:.0f}bp limit.'.ljust(78) + reset)
    print(banner + ' ' * 78 + reset)

    if reconstructed:
        d_alpha = np.array([abs(validation_results[k]['alpha'] - vcub_snapshot[k]['ref_alpha'])
                            for k in validation_results])
        d_nu = np.array([abs(validation_results[k]['nu'] - vcub_snapshot[k]['ref_nu'])
                         for k in validation_results])
        d_rho = np.array([abs(validation_results[k]['rho'] - vcub_snapshot[k]['ref_rho'])
                          for k in validation_results])
        print(f"\nparameter recovery vs VCUB reference, max absolute error:")
        print(f"  alpha {d_alpha.max():.2e}   nu {d_nu.max():.2e}   rho {d_rho.max():.2e}")

In [ ]:
if vcub_snapshot:
    plot_keys = VALIDATION_PLOT_SLICES or list(vcub_snapshot)
    plot_keys = [k for k in plot_keys if k in vcub_snapshot]
    ncols = min(3, len(plot_keys))
    nrows = int(np.ceil(len(plot_keys) / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)
    for ax, key in zip(axes.flat, plot_keys):
        sl = vcub_snapshot[key]
        res = validation_results[key]
        fine = np.linspace(sl['strikes'][0], sl['strikes'][-1], 200)
        fit = hagan_normal_vol(fine, sl['atm_forward'], sl['expiry_years'],
                               res['alpha'], res['beta'], res['nu'], res['rho'])
        ax.plot(sl['strike_offsets_bp'], sl['implied_vols'] * 10000, 'o',
                color='#00d2ff', markersize=8, label='VCUB', zorder=5)
        ax.plot((fine - sl['atm_forward']) * 10000, fit * 10000, '-',
                color='#4ade80', linewidth=2.5, label='SABR fit')
        ax.axvline(0, color='#888', linestyle=':', alpha=0.4)
        ax.set_title(f"{key[0]} into {key[1]}  (RMSE {res['rmse']*10000:.2f}bp)",
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('strike offset (bp)')
        ax.set_ylabel('normal vol (bp)')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    for ax in axes.flat[len(plot_keys):]:
        ax.axis('off')

    date = next(iter(vcub_snapshot.values()))['date']
    plt.suptitle(f'VCUB market vols vs SABR fit, snapshot {date} [REAL VCUB DATA]',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('no VCUB snapshot, nothing to plot')

A single day snapshot validates the method on one cross section. It does not establish a time
series, it says nothing about parameter stability day to day, and because each slice is fitted
independently the resulting surface is not arbitrage free across expiries. Fitting well at every
slice and being arbitrage free across the grid are different properties, and only the first is
tested here.

## Fallback data for non-VCUB slices

The snapshot covers 13 expiries by 7 tenors. Anything off that grid falls back in order: FRED for a
real ATM level with a synthetic smile built around it, then fully synthetic if FRED is unreachable.
Both are tagged and neither is presented as market observed. The two slices below are off grid on
purpose, to exercise the fallback path.

In [ ]:
fred_level = fred_atm_level()

FALLBACK_SLICES = [('4Yr', '7Yr'), ('8Yr', '12Yr')]

fallback_slices = {}
for expiry_label, tenor_label in FALLBACK_SLICES:
    if (expiry_label, tenor_label) in vcub_snapshot:
        continue
    fallback_slices[(expiry_label, tenor_label)] = build_fallback_slice(
        parse_term(expiry_label), parse_term(tenor_label), fred_level)

print()
for key, sl in fallback_slices.items():
    print(f"{key[0]:>5} into {key[1]:<5} source={sl['source']:<10} {sl['quote_kind']}")
    print(f"      atm forward {sl['atm_forward']*100:.3f}%  atm vol {sl['implied_vols'][4]*100:.2f}% lognormal")

## Full surface calibration with source tags

Every slice, VCUB and fallback, calibrated and stored with its source, vol convention and error.
Downstream sections read the source tag off each result rather than assuming anything.

In [ ]:
all_slices = {**vcub_snapshot, **fallback_slices}
all_results = {}

for key, sl in all_slices.items():
    if key in validation_results:
        res = dict(validation_results[key])
    else:
        res = calibrate_sabr(sl['strikes'], sl['atm_forward'], sl['expiry_years'],
                             sl['implied_vols'], beta=0.5, vol_model=sl['vol_model'])
    res.update({'source': sl['source'], 'date': sl['date'], 'quote_kind': sl['quote_kind'],
                'expiry_years': sl['expiry_years'], 'tenor_years': sl['tenor_years'],
                'atm_forward': sl['atm_forward']})
    all_results[key] = res

by_source = {}
for res in all_results.values():
    by_source[res['source']] = by_source.get(res['source'], 0) + 1

print('calibrated slices by source:')
for src, count in sorted(by_source.items()):
    print(f"  {count:>3}  {source_label(src)}")

## SABR vs market at a strike

Pick a slice and a strike. The comparison reports the SABR vol at that strike, the nearest quoted
vol from the slice, and the difference. The header states whether the reference is real VCUB or a
synthetic stand in.

In [ ]:
# sabr vol at a strike against the nearest available reference quote
def compare_at_strike(key, strike, is_payer=True):
    if key not in all_results:
        print(f'no slice {key}')
        return None

    sl, res = all_slices[key], all_results[key]
    vol_fn = hagan_normal_vol if res['vol_model'] == 'normal' else hagan_implied_vol
    unit = 'bp' if res['vol_model'] == 'normal' else '%'
    scale = 10000 if res['vol_model'] == 'normal' else 100

    sabr_vol = float(vol_fn(strike, sl['atm_forward'], res['expiry_years'],
                            res['alpha'], res['beta'], res['nu'], res['rho']))
    idx = int(np.argmin(np.abs(sl['strikes'] - strike)))
    ref_vol = float(sl['implied_vols'][idx])

    if res['source'] == 'VCUB':
        heading = 'SABR vs real market (VCUB)'
    else:
        heading = f"SABR vs synthetic reference ({source_label(res['source'])})"

    print(heading)
    print(f"  slice          {key[0]} into {key[1]}   source {res['source']}   {res['quote_kind']}")
    print(f"  atm forward    {sl['atm_forward']*100:.4f}%")
    print(f"  strike         {strike*100:.4f}%  ({(strike - sl['atm_forward'])*10000:+.1f}bp from ATM)")
    print(f"  SABR vol       {sabr_vol*scale:.3f}{unit}  ({res['vol_model']} convention)")
    print(f"  nearest quote  {ref_vol*scale:.3f}{unit}  at strike {sl['strikes'][idx]*100:.4f}%")
    print(f"  difference     {(sabr_vol - ref_vol)*scale:+.3f}{unit}")
    return {'sabr_vol': sabr_vol, 'ref_vol': ref_vol, 'source': res['source']}


COMPARE_KEY = ('5Yr', '10Yr') if ('5Yr', '10Yr') in all_results else next(iter(all_results))
compare_strike = all_slices[COMPARE_KEY]['atm_forward'] + 0.0050

compare_at_strike(COMPARE_KEY, compare_strike)

## Pricing

The earlier cells price one unit of annuity. A real number needs a real annuity.

The curve is bootstrapped from the SOFR par curve of the snapshot date, so discounting and vols come
from the same day. Pillars inside the first coupon date are treated as simple compounded deposits,
the rest are bootstrapped on a semi-annual grid, and discount factors interpolate log-linearly.
The annuity is the sum of Actual/360 day count weighted discount factors over the fixed leg, the
forward swap rate is derived from the curve rather than assumed, and Black-76 runs under that
annuity measure. If the snapshot curve is missing the code tries live FRED par rates, and failing
that falls back to a flat discount annuity, labelled as approximate in the output.

Forward swap rates from the bootstrapped curve are printed against the snapshot's own forwards as a
cross check. They come from the same SOFR curve by different routes, so agreement to a few basis
points is the expected result and a large gap would mean the bootstrap is wrong.

In [ ]:
par_rates = load_par_curve('data/sofr_curve.csv')
curve_source = 'snapshot SOFR curve'

if not par_rates:
    par_rates = fetch_fred_par_curve()
    curve_source = 'live FRED par rates'

curve = bootstrap_discount_curve(par_rates) if par_rates else None

if curve is not None:
    times, dfs = curve
    print(f"\nbootstrapped {len(times)-1} pillars out to {times[-1]:.1f}Y from the {curve_source}")
    print(f"  df(1Y) = {float(discount_factor(curve, 1)):.4f}   "
          f"df(10Y) = {float(discount_factor(curve, 10)):.4f}   "
          f"df(30Y) = {float(discount_factor(curve, 30)):.4f}")
    print('\nforward cross check, curve against the snapshot quotes:')
    for k in [('1Yr', '10Yr'), ('5Yr', '10Yr'), ('10Yr', '10Yr'), ('30Yr', '30Yr')]:
        if k not in vcub_snapshot:
            continue
        sl_k = vcub_snapshot[k]
        f_curve = forward_swap_rate(curve, sl_k['expiry_years'], sl_k['tenor_years'])
        tail = sl_k['expiry_years'] + sl_k['tenor_years']
        note = '  (extrapolated past the last curve pillar)' if tail > curve_max_time(curve) else ''
        print(f"  {k[0]:>5} into {k[1]:<5} curve {f_curve*100:.4f}%   "
              f"snapshot {sl_k['atm_forward']*100:.4f}%   "
              f"gap {(f_curve - sl_k['atm_forward'])*10000:+6.1f}bp{note}")
else:
    print('\nno curve available, pricing falls back to a flat discount annuity')

PRICE_KEY = ('5Yr', '10Yr') if ('5Yr', '10Yr') in all_results else next(iter(all_results))
res = all_results[PRICE_KEY]
sl = all_slices[PRICE_KEY]
notional = 100_000_000

expiry_y, tenor_y = res['expiry_years'], res['tenor_years']
strike = sl['atm_forward'] + 0.0050

if curve is not None:
    fwd_curve = forward_swap_rate(curve, expiry_y, tenor_y)
    ann = annuity_from_curve(curve, expiry_y, tenor_y)
    annuity_note = f'bootstrapped {curve_source}, ACT/360 semi-annual'
else:
    fwd_curve = sl['atm_forward']
    ann = flat_annuity(sl['atm_forward'], tenor_y)
    annuity_note = 'APPROXIMATE ANNUITY, flat discount, no market curve'
    curve_source = 'none'

quote = price_swaption(sl['atm_forward'], strike, expiry_y, tenor_y,
                       res['alpha'], res['beta'], res['nu'], res['rho'],
                       curve=curve, notional=notional)
greeks = compute_greeks(sl['atm_forward'], strike, expiry_y, res['alpha'],
                        res['beta'], res['nu'], res['rho'])

print()
print(f"  instrument      {PRICE_KEY[0]} payer swaption into {PRICE_KEY[1]} swap")
print(f"  vol source      {source_label(res['source'])}")
print(f"  snapshot date   {res['date']}")
print(f"  vol quote kind  {res['quote_kind']}")
print(f"  annuity         {ann:.4f}   [{annuity_note}]")
print(f"  curve source    {curve_source}")
print(f"  fwd from curve  {fwd_curve*100:.4f}%")
print(f"  fwd from vols   {sl['atm_forward']*100:.4f}%  (snapshot ATM forward)")
print(f"  strike          {strike*100:.4f}%  (ATM + 50bp)")
print(f"  SABR lognormal vol  {quote['vol']*100:.2f}%")
print(f"  price (unit annuity) {quote['unit_price']:.6f}")
print(f"  price               ${quote['price']:,.0f}  on ${notional:,.0f} notional")
print(f"  vega per 1bp        ${greeks['vega'] * ann * notional:,.0f}")

if res['source'] == 'VCUB':
    print('\n  closest thing here to a real market price: vol from the VCUB snapshot,')
    print(f'  discounting from a curve bootstrapped off the {curve_source}.'
          if curve is not None else
          '  but with an approximate flat annuity since no curve was available.')
    if curve is not None:
        gap_bp = (fwd_curve - sl['atm_forward']) * 10000
        print(f"  vols and curve are both from {res['date']}, so this is internally consistent.")
        print(f"  the two forwards agree to {gap_bp:+.1f}bp, which is the bootstrap cross check.")
        print('  the option is priced on the snapshot forward, the annuity on the curve.')
else:
    print(f"\n  NOT a market price. the vol input is {source_label(res['source'])}.")

## Summary

One row per slice, VCUB first. Source, parameters, fit error and price in one view, so the
trustworthy rows and the best effort rows sit side by side and are told apart at a glance.

In [ ]:
rows = []
for key, res in all_results.items():
    sl = all_slices[key]
    priced = price_swaption(sl['atm_forward'], sl['atm_forward'],
                            res['expiry_years'], res['tenor_years'],
                            res['alpha'], res['beta'], res['nu'], res['rho'], curve=curve)
    rows.append({
        'expiry': key[0], 'tenor': key[1], 'source': res['source'],
        'alpha': res['alpha'], 'nu': res['nu'], 'rho': res['rho'],
        'rmse_bp': res['rmse'] * 10000, 'price': priced['unit_price'],
        'flag': 'extrap' if (curve is not None and
                             res['expiry_years'] + res['tenor_years'] > curve_max_time(curve)) else '',
        'order': (0 if res['source'] == 'VCUB' else 1 if res['source'] == 'FRED' else 2,
                  res['expiry_years'], res['tenor_years']),
    })

rows.sort(key=lambda r: r['order'])

header = (f"{'expiry':>7} {'tenor':>6} {'source':>10} {'alpha':>9} {'nu':>8} {'rho':>8} "
          f"{'rmse(bp)':>9} {'price':>10} {'flag':>8}")
print(header)
print('-' * len(header))
for r in rows:
    print(f"{r['expiry']:>7} {r['tenor']:>6} {r['source']:>10} {r['alpha']:>9.6f} "
          f"{r['nu']:>8.4f} {r['rho']:>8.4f} {r['rmse_bp']:>9.3f} {r['price']:>10.6f} "
          f"{r['flag']:>8}")

print()
print('price is ATM, per unit notional, under the '
      + ('bootstrapped' if curve is not None else 'approximate flat') + ' annuity.')
print('VCUB rows are market observed vols. FRED rows are a real ATM level with a synthetic')
print('smile. synthetic rows are generated end to end and are not market data.')
print("flag 'extrap' marks slices whose last payment falls beyond the curve's final pillar,")
print('where the annuity relies on flat forward extrapolation.')

## Vol surface in three dimensions

The VCUB snapshot across expiries at a fixed swap tenor, fitted slice by slice and interpolated
across the expiry axis. Cyan points are the input quotes, the mesh is the SABR fit.

In [ ]:
from scipy.interpolate import RegularGridInterpolator

SURFACE_TENOR = '10Yr'
surface_keys = [k for k in vcub_snapshot if k[1] == SURFACE_TENOR]

if len(surface_keys) >= 2:
    surface_keys.sort(key=lambda k: parse_term(k[0]))
    surface_exp = [parse_term(k[0]) for k in surface_keys]

    k_min = max(vcub_snapshot[k]['strikes'].min() for k in surface_keys)
    k_max = min(vcub_snapshot[k]['strikes'].max() for k in surface_keys)
    grid_strikes = np.linspace(k_min, k_max, 80)

    vol_matrix = np.zeros((len(surface_keys), len(grid_strikes)))
    for i, key in enumerate(surface_keys):
        sl, res = vcub_snapshot[key], all_results[key]
        vol_matrix[i, :] = hagan_normal_vol(grid_strikes, sl['atm_forward'], res['expiry_years'],
                                            res['alpha'], res['beta'], res['nu'], res['rho']) * 10000

    fine_exp = np.linspace(min(surface_exp), max(surface_exp), 60)
    interp = RegularGridInterpolator((np.array(surface_exp), grid_strikes), vol_matrix,
                                     method='linear', bounds_error=False, fill_value=None)
    strike_mesh, expiry_mesh = np.meshgrid(grid_strikes, fine_exp)
    vol_smooth = interp(np.column_stack([expiry_mesh.ravel(), strike_mesh.ravel()])).reshape(expiry_mesh.shape)

    fig = plt.figure(figsize=(14, 9))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(strike_mesh * 100, expiry_mesh, vol_smooth,
                           cmap='plasma', edgecolor='none', alpha=0.92)

    for key in surface_keys:
        sl = vcub_snapshot[key]
        ax.scatter(sl['strikes'] * 100, np.full_like(sl['strikes'], parse_term(key[0])),
                   sl['implied_vols'] * 10000, color='#00d2ff', s=25,
                   edgecolors='white', linewidth=0.5, zorder=10)

    ax.set_xlabel('strike (%)', labelpad=10)
    ax.set_ylabel('option expiry (years)', labelpad=10)
    ax.set_zlabel('normal vol (bp)', labelpad=10)
    date = vcub_snapshot[surface_keys[0]]['date']
    ax.set_title(f'Swaption normal vol surface, {SURFACE_TENOR} tenor\n'
                 f'[REAL VCUB DATA, snapshot {date}]', fontsize=13, fontweight='bold', pad=20)
    ax.view_init(elev=25, azim=-55)
    for pane in (ax.xaxis, ax.yaxis, ax.zaxis):
        pane.pane.fill = False
    fig.colorbar(surf, ax=ax, shrink=0.55, aspect=12, pad=0.1, label='normal vol (bp)')
    plt.tight_layout()
    plt.show()
else:
    print(f'fewer than two VCUB slices at {SURFACE_TENOR} tenor, surface plot skipped')

## Limitations

Each expiry and tenor is calibrated independently, so a good fit everywhere does not make the
surface arbitrage free across expiries, and nothing here checks calendar or butterfly conditions on
the fitted grid. There is no shift term, so the model is unusable at or below zero rates. Hagan's
expansion degrades at very long expiries and far strikes, where Obloj's correction or a direct
normal SABR parameterisation does better. The curve is a single SOFR discount curve with no basis
or multi-curve adjustment, and swaps maturing past its last pillar at 30Y rely on flat forward
extrapolation, so those annuities are indicative rather than desk numbers.

One day of VCUB is enough to check that the code reproduces a real smile. It is not enough to say
anything about parameter stability, term structure dynamics, or how the fit holds up through a
selloff.